# NpuKit — MNIST tiny-ViT (PYNQ-Z2)

Geometry: resize **28→16**, patch **4**, pair-average → **T=8**, **D=8**.

Train on the Docker host (torch):
```bash
python3 host/train_vit_mnist.py
```
Then copy `vit_mnist_weights.npz`, `mnist_sample.npz`, and this notebook to the board.

Float ~81%, quantized ref ~79% after scale calibration + STE QAT. This notebook checks **ref vs board** match and batch accuracy.

In [1]:
import importlib
import sys

BIT = "/home/xilinx/jupyter_notebooks/npukit.bit"
sys.path.insert(0, "/home/xilinx/jupyter_notebooks")

import npukit_vit_mnist as vit

importlib.reload(vit)
print("T", vit.VIT_T, "D", vit.VIT_D)
print("weights", vit.DEFAULT_WEIGHTS, "exists", vit.DEFAULT_WEIGHTS.exists())
print("sample", vit.DEFAULT_SAMPLE, "exists", vit.DEFAULT_SAMPLE.exists())

T 8 D 8
weights /home/xilinx/jupyter_notebooks/vit_mnist_weights.npz exists True
sample /home/xilinx/jupyter_notebooks/mnist_sample.npz exists True


## Offline ref (trained weights + real MNIST sample)

In [2]:
rc = vit.run_vit_smoke(bit_path=None, seed=0, n=64)
assert rc == 0
print("ref-only return", rc)

loaded MNIST sample from /home/xilinx/jupyter_notebooks/mnist_sample.npz (n=16)
=== MNIST tiny-ViT smoke ===
IMG=16 PATCH=4 T=8 D=8 classes=10
scales ACT/W/P=17.59/94.27/133.51
weights=/home/xilinx/jupyter_notebooks/vit_mnist_weights.npz

--- image[0] label=6 ---
--- ref ---
ref pred=6 logits_q12[:4]=[19024, -16757, 21384, -14363]

--- image[1] label=8 ---
--- ref ---
ref pred=8 logits_q12[:4]=[15089, -20583, -4432, 4570]

--- image[2] label=3 ---
--- ref ---
ref pred=3 logits_q12[:4]=[-2033, -9081, 19684, 20272]

--- image[3] label=0 ---
--- ref ---
ref pred=0 logits_q12[:4]=[28261, -31379, 5153, 5610]

--- image[4] label=2 ---
--- ref ---
ref pred=2 logits_q12[:4]=[-14496, 24153, 38192, 13622]

--- image[5] label=9 ---
--- ref ---
ref pred=9 logits_q12[:4]=[-5267, -13644, -22426, -2745]

--- image[6] label=0 ---
--- ref ---
ref pred=0 logits_q12[:4]=[27624, -37856, -6131, 13177]

--- image[7] label=2 ---
--- ref ---
ref pred=2 logits_q12[:4]=[-10660, 12693, 44141, 15141]

--- image[8

## Board: ref vs FPGA + batch accuracy

In [3]:
rc = vit.run_vit_smoke(bit_path=BIT, seed=0, n=64)
assert rc == 0, "ViT smoke failed"
print("board vit return", rc)

loaded MNIST sample from /home/xilinx/jupyter_notebooks/mnist_sample.npz (n=16)
=== MNIST tiny-ViT smoke ===
IMG=16 PATCH=4 T=8 D=8 classes=10
scales ACT/W/P=17.59/94.27/133.51
weights=/home/xilinx/jupyter_notebooks/vit_mnist_weights.npz


Using AXI DMA transport (/home/xilinx/jupyter_notebooks/npukit.bit)
ID=0x4E50554B version=0x00000300 features=0x00000003

--- image[0] label=6 ---
--- ref ---
ref pred=6 logits_q12[:4]=[19024, -16757, 21384, -14363]
--- FPGA ---
hw  pred=6 logits_q12[:4]=[19024, -16757, 21384, -14363]
tokens: PASS  max|err|=0  tol=512
block.y_out: PASS  max|err|=242  tol=1024
logits: PASS  max|err|=0  tol=1024

--- image[1] label=8 ---
--- ref ---
ref pred=8 logits_q12[:4]=[15089, -20583, -4432, 4570]
--- FPGA ---
hw  pred=8 logits_q12[:4]=[15089, -20583, -4432, 4570]
tokens: PASS  max|err|=0  tol=512
block.y_out: PASS  max|err|=345  tol=1024
logits: PASS  max|err|=0  tol=1024

--- image[2] label=3 ---
--- ref ---
ref pred=3 logits_q12[:4]=[-2033, -9081, 19684, 20272]
--- FPGA ---
hw  pred=3 logits_q12[:4]=[-2033, -9081, 19684, 20272]
tokens: PASS  max|err|=0  tol=512
block.y_out: PASS  max|err|=277  tol=1024
logits: PASS  max|err|=0  tol=1024

--- image[3] label=0 ---
--- ref ---
ref pred=0 logits_q12